# Capítulo 10 — Trabalhando com Dados

**Livro:** começa na página **179** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import math, random, re
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
random.seed(0)
print("Pronto. Aqui é tudo do zero: só Python puro (e matplotlib para gráficos).")

## Explorando Seus Dados · PDF p. 180

#### Exercício: Explorar uma e duas dimensões

Gere 1.000 alturas (normal 170, 10) e pesos (peso = 0,9·altura − 90 + ruído). Mostre o histograma das alturas e a dispersão altura × peso.

In [ ]:
alt = [random.gauss(170, 10) for _ in range(1000)]
peso = [0.9 * a - 90 + random.gauss(0, 5) for a in alt]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(alt, bins=30); ax[0].set_title("altura")
ax[1].scatter(alt, peso, s=5); ax[1].set_title("altura × peso"); plt.show()

## Limpando e Transformando · PDF p. 186

#### Exercício: Limpar linhas sujas

Converta `[["2024-01-02", "AAA", "10.5"], ["2024-01-03", "BBB", "n/a"], ["data", "AAA", "9"]]` para (datetime.date, str, float), descartando as linhas que não puderem ser convertidas.

In [ ]:
from datetime import date
def converte(linha):
    try:
        return date.fromisoformat(linha[0]), linha[1], float(linha[2])
    except ValueError:
        return None
brutas = [["2024-01-02", "AAA", "10.5"], ["2024-01-03", "BBB", "n/a"], ["data", "AAA", "9"]]
print([r for r in map(converte, brutas) if r])

## Manipulando Dados · PDF p. 189

#### Exercício: Agrupar e resumir sem pandas

Com uma lista de dicionários de vendas `{"loja", "mes", "valor"}` (gere 50 aleatórias), calcule o total por loja e a maior venda de cada mês usando `defaultdict`.

In [ ]:
vendas = [{"loja": random.choice("XYZ"), "mes": random.randint(1, 3), "valor": round(random.uniform(10, 500), 2)} for _ in range(50)]
total = defaultdict(float); maior = defaultdict(float)
for v in vendas:
    total[v["loja"]] += v["valor"]
    maior[v["mes"]] = max(maior[v["mes"]], v["valor"])
print({k: round(t, 2) for k, t in total.items()}, dict(maior))

## Redimensionando · PDF p. 193

#### Exercício: Padronização (z-score)

Distâncias dependem da unidade. Com 3 pessoas (altura em cm, peso em kg), calcule as distâncias entre pares antes e depois de padronizar cada coluna (média 0, desvio 1).

In [ ]:
pessoas = [[160, 60], [170, 85], [175, 62]]
def padroniza(dados):
    cols = list(zip(*dados))
    m = [sum(c) / len(c) for c in cols]
    s = [math.sqrt(sum((x - mi) ** 2 for x in c) / (len(c) - 1)) for c, mi in zip(cols, m)]
    return [[(x - mi) / si for x, mi, si in zip(p, m, s)] for p in dados]
for dados in (pessoas, padroniza(pessoas)):
    print([round(math.dist(dados[i], dados[j]), 2) for i, j in [(0, 1), (0, 2), (1, 2)]])

## Redução da Dimensionalidade · PDF p. 195

#### Exercício: Primeiro componente principal por gradiente

Gere 200 pontos 2D alongados na direção (1, 1). Centralize e encontre a direção de maior variância maximizando a variância projetada com subida de gradiente (normalizando o vetor a cada passo).

In [ ]:
pts = [[t + random.gauss(0, 0.3), t + random.gauss(0, 0.3)] for t in [random.gauss(0, 2) for _ in range(200)]]
m = [sum(c) / len(c) for c in zip(*pts)]
X = [[x - m[0], y - m[1]] for x, y in pts]
w = [random.random(), random.random()]
for _ in range(500):
    g = [sum(2 * (x[0] * w[0] + x[1] * w[1]) * x[i] for x in X) / len(X) for i in range(2)]
    w = [wi + 0.01 * gi for wi, gi in zip(w, g)]
    n = math.hypot(*w); w = [wi / n for wi in w]
print([round(x, 3) for x in w])  # ~(0,707; 0,707)